# Experimento: Ejecutar HPT con Búsqueda en Cuadrícula para construir un modelo Naive Bayes

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import string
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import BernoulliNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

import nltk
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import word_tokenize

from src import utils

# Parámetros

In [3]:
RND_SEED = 123
PCT_TEST = 0.2
K_FOLD = 3

EXPERIMENT = "exp01_hpt_nb"

# Paths
path_interim = os.path.join("data", "interim")
path_experiment =  os.path.join(path_interim, EXPERIMENT)

# Input
file_train = "train.csv"

# Output
file_exp = "df_exp_summary.csv"

In [4]:


utils.create_or_clean_folder(path_experiment)

Creating the folder: data\interim\exp01_hpt_nb


# Cargar datos

In [5]:
path_data_train = os.path.join(path_interim, file_train)

df_train = pd.read_csv(path_data_train)
df_train.head()

,x_text,y_is_nf
0,Respuestas coherentes e idénticas ante entrada...,0
1,Gestión de usuarios: Todos los administradores...,0
2,Añadir numero de una revista. Para ello debemo...,0
3,Un usuario registrado visualiza la tabla de en...,0
4,Como usuario quiero poder ordenar las listas d...,0


# Construir Tubería

In [6]:
# Celda auxiliar: Tokenización y lematización en español
import typing
import string


def tokenizer_stemmer_es(text) -> typing.List[str]:
    stopword_es = nltk.corpus.stopwords.words('spanish')
    stemmer = SnowballStemmer("spanish")

    clean_words = [word for word in word_tokenize(text) if word not in string.punctuation and word.lower() not in stopword_es] # list[str]
    return [stemmer.stem(word) for word in clean_words]  # list[str]


stopwords_es = nltk.corpus.stopwords.words('spanish')

example = df_train.loc[0, "x_text"]
ex_stem = tokenizer_stemmer_es(example)

print(f"{example=}")
print(f"{ex_stem=}")


example='Respuestas coherentes e idénticas ante entradas de audio o texto: Los usuarios tienen la posibilidad de escuchar la respuesta mediante voz, esta ha de ser entendida e idéntica a la respuesta por escrito.'
ex_stem=['respuest', 'coherent', 'ident', 'entrad', 'audi', 'text', 'usuari', 'posibil', 'escuch', 'respuest', 'mediant', 'voz', 'ser', 'entend', 'ident', 'respuest', 'escrit']


In [7]:
# Elige instancias apropiadas de XXXVectorizer y BernoulliXXX
tfbin_unigrams = CountVectorizer(
    strip_accents="ascii",
    lowercase=True,
    tokenizer=tokenizer_stemmer_es,
    ngram_range=(1, 1),
    binary=True,
)


clf_nbber = BernoulliNB()

# Crear la tubería
skl_pl = Pipeline([
    ('fte', tfbin_unigrams),
    ('clf', clf_nbber)
])

# Validación cruzada del modelo

Usa el objeto Búsqueda en CuadrículaCV pero con una única configuración,
para mantener el esquema de experimentos fácilmente comparable.
También podrías usar otros métodos de CV

Recuerda usar siempre el mismo número de pliegues CV y la misma métrica CV en 
¡cada experimento!


In [8]:
X_train = df_train['x_text']
y_train = df_train['y_is_nf']

# Change at will
param_grid = {
    # pasodeltubo__parámetro: [lista de valores de parámetros]
    # Verifica en la documentación qué parámetros vale la pena probar
    # y qué valores esperan
    'fte__max_features': [None, 64, 128, 256],
    'fte__max_df': [0.5, 0.75, 0.95],
    'fte__min_df': [1, 3, 5],
}

grid_search = GridSearchCV(
    skl_pl,
    param_grid,
    cv=K_FOLD,  # mantener el mismo número de pliegues en todo el proyecto
    scoring='f1',  # mantener la misma función de puntuación (métrica cv) en todo el proyecto
    n_jobs=-1
    )

# Ajustar Búsqueda en CuadrículaCV en los datos de entrenamiento
grid_search.fit(X_train, y_train)
print(f"{grid_search.best_score_=}")

grid_search.best_score_=np.float64(0.7679748822605966)


c:\Users\usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [9]:
df_exp_summary = pd.DataFrame(
    grid_search.cv_results_
)

df_exp_summary["experiment_id"] = EXPERIMENT
df_exp_summary 

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_fte__max_df,param_fte__max_features,param_fte__min_df,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,experiment_id
0,0.376776,0.108278,0.210442,0.039959,0.50,None,1,"{'fte__max_df': 0.5, 'fte__max_features': None...",0.529412,0.684211,0.615385,0.609669,0.063325,34,exp01_hpt_nb
1,0.336934,0.103631,0.180520,0.053305,0.50,None,3,"{'fte__max_df': 0.5, 'fte__max_features': None...",0.711111,0.769231,0.777778,0.752707,0.029619,19,exp01_hpt_nb
2,0.472461,0.009679,0.241844,0.006938,0.50,None,5,"{'fte__max_df': 0.5, 'fte__max_features': None...",0.750000,0.769231,0.777778,0.765670,0.011616,6,exp01_hpt_nb
3,0.369721,0.095408,0.208321,0.031207,0.50,64,1,"{'fte__max_df': 0.5, 'fte__max_features': 64, ...",0.716981,0.734694,0.734694,0.728790,0.008350,25,exp01_hpt_nb
4,0.393525,0.073332,0.224251,0.003685,0.50,64,3,"{'fte__max_df': 0.5, 'fte__max_features': 64, ...",0.716981,0.734694,0.734694,0.728790,0.008350,25,exp01_hpt_nb
5,0.378238,0.033911,0.224514,0.014679,0.50,64,5,"{'fte__max_df': 0.5, 'fte__max_features': 64, ...",0.716981,0.734694,0.734694,0.728790,0.008350,25,exp01_hpt_nb
6,0.455398,0.022858,0.231152,0.011109,0.50,128,1,"{'fte__max_df': 0.5, 'fte__max_features': 128,...",0.750000,0.754717,0.784314,0.763010,0.015186,8,exp01_hpt_nb
7,0.482044,0.002017,0.230378,0.008262,0.50,128,3,"{'fte__max_df': 0.5, 'fte__max_features': 128,...",0.750000,0.754717,0.777778,0.760832,0.012137,10,exp01_hpt_nb
8,0.469502,0.006064,0.237005,0.004107,0.50,128,5,"{'fte__max_df': 0.5, 'fte__max_features': 128,...",0.750000,0.754717,0.792453,0.765723,0.018998,5,exp01_hpt_nb
9,0.456182,0.000740,0.241605,0.014324,0.50,256,1,"{'fte__max_df': 0.5, 'fte__max_features': 256,...",0.739130,0.769231,0.777778,0.762046,0.016575,9,exp01_hpt_nb


# Diagnosticar el modelo

In [10]:
# Verificar dimensiones de DTM
skl_pl_fitted = grid_search.best_estimator_  
# Solo un modelo se ajusta, ya que solo una configuración HPT se pasa

# Acceder a la parte CountVectorizer de la tubería
skl_pl_fte = skl_pl_fitted.named_steps['fte']

# Obtener DTM con transform()
dtm_train = skl_pl_fte.transform(X_train)
print(f"{dtm_train.shape=}")  # columnas: Número de términos en el vocabulario

dtm_train.shape=(311, 215)


In [11]:
# Verificar predicciones de entrenamiento y puntuación

y_hats_train = grid_search.best_estimator_.predict(X_train)  # obtener predicciones con predict()
f1_score_train = f1_score(
    y_true=y_train,
    y_pred=y_hats_train
)

print(f"{f1_score_train=}")  # ¿Es comparable con la métrica CV?

f1_score_train=0.8170731707317073


# Escribir resultados de experimentos

In [12]:
df_exp_summary.to_csv(
    os.path.join(path_experiment, file_exp),
    index=False
)

# otros resultados de experimentos y artefactos podrían ser útiles